# event_datasets.duckdb Exploration

This notebook helps you inspect and explore the DuckDB database at `./data/event_datasets.duckdb`.

In [2]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [3]:
# Resolve the project root from the notebook location.
project_root = Path.cwd().resolve().parent
db_path = project_root / "data" / "event_datasets.duckdb"

print(f"Project root: {project_root}")
print(f"DB path: {db_path}")
print(f"Exists: {db_path.exists()}")

Project root: /Users/6161138/uv_projects/teamnl-events-dataset
DB path: /Users/6161138/uv_projects/teamnl-events-dataset/data/event_datasets.duckdb
Exists: True


In [4]:
con = duckdb.connect(str(db_path), read_only=True)
con.execute("SELECT current_database()").fetchone()

('event_datasets',)

## Discover Schema

In [5]:
# List tables and views in the database.
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,event_datasets,main,kg_datasets,"[dataset_id, title, alt_title, publication_dat...","[VARCHAR, VARCHAR, VARCHAR, DATE, DATE, DATE, ...",False
1,event_datasets,main,kg_variables,"[variable_id, dataset_id, variable_name, label...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


In [6]:
# For each table, inspect schema and row count.
tables_df = con.execute("SHOW ALL TABLES").df()
table_names = tables_df["name"].dropna().tolist() if "name" in tables_df.columns else []

summary = []
for table in table_names:
    count = con.execute(f"SELECT COUNT(*) FROM \"{table}\"").fetchone()[0]
    summary.append({"table": table, "rows": count})

pd.DataFrame(summary).sort_values("rows", ascending=False).reset_index(drop=True)

,table,rows
0,kg_variables,63136
1,kg_datasets,578


In [9]:
# Pick one table to preview. Change selected_table as needed.
selected_table = table_names[1] if table_names else None
selected_table

'kg_variables'

In [10]:
if selected_table:
    display(con.execute(f"DESCRIBE \"{selected_table}\"").df())
    display(con.execute(f"SELECT * FROM \"{selected_table}\" LIMIT 20").df())
else:
    print("No tables found.")

,column_name,column_type,null,key,default,extra
0,variable_id,VARCHAR,YES,None,None,None
1,dataset_id,VARCHAR,YES,None,None,None
2,variable_name,VARCHAR,YES,None,None,None
3,label_,VARCHAR,YES,None,None,None
4,vocab_label,VARCHAR,YES,None,None,None
5,description,VARCHAR,YES,None,None,None
6,data_type,VARCHAR,YES,None,None,None
7,definition,VARCHAR,YES,None,None,None
8,num_predicates,VARCHAR,YES,None,None,None
9,tijdstip_predicates,VARCHAR[],YES,None,None,None


,variable_id,dataset_id,variable_name,label_,vocab_label,description,data_type,definition,num_predicates,tijdstip_predicates,valid_from,is_tijdstip
0,https://kg.odissei.nl/.well-known/genid/711e5e...,https://doi.org/10.57934/0B01E410804102B1,SVOVRGEigNivb,Geeft aan hoe men het eigen niveau inschat op ...,Geeft aan hoe men het eigen niveau inschat op ...,Binnen de vraag EigNiv kan de respondent voor ...,Integer,Het door een persoon geschat niveau van werkge...,14,[],2016-09-05,False
1,https://kg.odissei.nl/.well-known/genid/f396e2...,https://doi.org/10.57934/0B01E41080787427,WNEVrgDeelLeid,Waaruit bestaat het grootste deel van uw werkz...,Waaruit bestaat het grootste deel van uw werkz...,NaN,Integer,De inhoud waaruit het grootste deel van de wer...,13,[],2019-01-01,False
2,https://kg.odissei.nl/.well-known/genid/d45695...,https://doi.org/10.57934/0B01E410804102B1,SVOREGSECJ,Sociaaleconomische categorie (jaarbasis) (V3),Sociaaleconomische categorie op jaarbasis voor...,Om de score op deze variabele te bepalen worde...,Integer,Classificatie van een persoon naar sociaalecon...,15,[],2003-01-01,False
3,https://kg.odissei.nl/.well-known/genid/bc201f...,https://doi.org/10.57934/0B01E410804102B1,SVOVRGTevrf,Wanneer u terugkijkt naar uw gevolgde opleidin...,Wanneer u terugkijkt naar uw gevolgde opleidin...,Binnen de vraag Tevr kan de respondent aangeve...,Integer,De mate van tevredenheid over een opleiding.,14,[],2014-09-15,False
4,https://kg.odissei.nl/.well-known/genid/4e9e7b...,https://doi.org/10.57934/0B01E410801F5B08,CCOVRGLft5,Leeftijd in jaren persoon 5,Leeftijd in jaren persoon 5,Leeftijd in jaren van vijfde persoon in het hu...,Integer,Leeftijd van een persoon,16,[],1904-12-31,False
5,https://kg.odissei.nl/.well-known/genid/a15fb0...,https://doi.org/10.57934/0B01E4108077580F,CCOVRGCode_mode,Interview Mode,Interview Mode,NaN,Integer,De manier waarop een respondent een enquête in...,14,[],2014-07-01,False
6,https://kg.odissei.nl/.well-known/genid/a69d00...,https://doi.org/10.57934/0B01E410801F7975,RINPERSOON,Samen met RINPERSOONS identificeert dit nummer...,Rinpersoon,NaN,String,Dit nummer identificeert een natuurlijk persoo...,13,[],1904-12-31,False
7,https://kg.odissei.nl/.well-known/genid/2d71a1...,https://doi.org/10.57934/0B01E4108045CBE3,ISCEDF2013DETAILEDFIELDAct,"ISCED detailed fields of study, actueel onderwijs","ISCED detailed fields of study, actueel onderwijs",NaN,String,Een opleidingsniveau van een persoon gebaseerd...,14,[],1981-01-01,False
8,https://kg.odissei.nl/.well-known/genid/4cbd16...,https://doi.org/10.57934/0B01E41080367507,SVOVRGBelKeuzed,Hoe belangrijk was de keuze voor de vervolgopl...,Hoe belangrijk was de keuze voor de vervolgopl...,Binnen de vraag BelKeuze kan de respondent aan...,String,De mate van belangrijkheid voor een specifiek ...,14,[],2016-09-05,False
9,https://kg.odissei.nl/.well-known/genid/f7c393...,https://doi.org/10.57934/0B01E41080367507,SVOVRGBelKeuzea,Hoe belangrijk was de keuze voor de vervolgopl...,Hoe belangrijk was de keuze voor de vervolgopl...,Binnen de vraag BelKeuze kan de respondent aan...,String,De mate van belangrijkheid voor een specifiek ...,14,[],2016-09-05,False


## Example Analysis Queries

In [ ]:
# Top 20 tables by row count (computed using metadata).
pd.DataFrame(summary).sort_values("rows", ascending=False).head(20)

In [ ]:
# Close the connection when done.
con.close()